In [27]:
import sys
sys.path.append('../src')
from data_loader import UniversalECGLoader
from tqdm.auto import tqdm # For a nice progress bar
import pandas as pd

Step 1: Create a Master List of All Records
First, let's combine the lists of valid records from both datasets into a single "master list." This will make it easy to loop through and process every single file in one go.

In [ ]:


# --- Initialize Loaders for Both Datasets ---
# (Update these paths if they are different)
afdb_path = '../data/raw/mit-bih-afdb/'
cinc_path = '../data/raw/physionet-challenge-2017/training/'

afdb_loader = UniversalECGLoader(afdb_path)
cinc_loader = UniversalECGLoader(cinc_path)

# --- Get Valid Records ---
# The get_valid_records() function caches the results, so this will be fast.
afdb_records = afdb_loader.get_valid_records()
cinc_records = cinc_loader.get_valid_records()

# --- Create the Master List ---
# We'll store tuples of (record_id, loader_object, dataset_name)
master_list = [(rec, afdb_loader, 'AFDB') for rec in afdb_records] + \
              [(rec, cinc_loader, 'CINC17') for rec in cinc_records]

print(f"✅ Created a master list with {len(master_list)} total records to process.")

Step 2: Build the Preprocessing and Segmentation Function
Now, let's create the function that will perform all the cleaning and windowing operations on a single ECG signal. This is where we address the inconsistencies we found during exploration.

In [29]:
# Cell 3: Configuration and Preprocessing Function
TARGET_FS = 250
WINDOW_SECONDS = 10
SAMPLES_PER_WINDOW = WINDOW_SECONDS * TARGET_FS  # 2500 samples

def preprocess_and_segment(signal, original_fs):
    """
    Takes a raw, single-lead ECG signal and returns a list of
    clean, standardized, 10-second windows.
    """
    # 1. Resample to the target frequency
    if original_fs != TARGET_FS:
        num_samples = int(len(signal) * (TARGET_FS / original_fs))
        signal = resample(signal, num_samples)

    # 2. Clean the resampled signal using NeuroKit2
    cleaned_signal = nk.ecg_clean(signal, sampling_rate=TARGET_FS)
    
    # 3. Segment the long signal into uniform windows
    windows = []
    for start in range(0, len(cleaned_signal) - SAMPLES_PER_WINDOW + 1, SAMPLES_PER_WINDOW):
        window = cleaned_signal[start : start + SAMPLES_PER_WINDOW]
        windows.append(window)
        
    if not windows:
        return np.array([])
        
    return np.array(windows)

Step 3: Handle the Labels (The Most Important Part)
This is the trickiest part. You need to get the correct label ("Normal", "AFib", "Other", or "Noisy") for each record. The two datasets store this information differently.

For the PhysioNet 2017 Challenge: The labels are in a single REFERENCE.csv file. We need to load this file into a dictionary for easy lookup.

For the MIT-BIH AFDB: The labels are stored in .atr annotation files, one for each record. The rhythm changes are in the aux_note field.

In [30]:
from pathlib import Path

# The main directory for the CINC 2017 dataset
search_dir = Path('../data/raw/physionet-challenge-2017/')

# --- For CINC 2017 ---
# Load the REFERENCE.csv file into a pandas DataFrame
cinc_labels_df = pd.read_csv(search_dir / 'REFERENCE-v3.csv', header=None, names=['record_id', 'label'])
# Convert it to a dictionary for fast lookup: {'A00001': 'N'}
cinc_labels_dict = dict(zip(cinc_labels_df['record_id'], cinc_labels_df['label']))

# --- For AFDB ---
# The labels are inside the annotation files, which our loader already reads.
# The common labels are '(N' for Normal and '(AFIB' for Atrial Fibrillation.

from pathlib import Path

def get_label(record_id, record_data, dataset_name):
    """
    Finds the correct label for a given record.
    """
    if dataset_name == 'CINC17':
        # ✅ FIX: Extract just the filename without path and extension
        clean_id = Path(record_id).stem  # 'A00001' from 'training/.../A00001.mat'
        label = cinc_labels_dict.get(clean_id, 'UNKNOWN')
        
        # Debug: Print if label not found
        if label == 'UNKNOWN':
            print(f"⚠️  No label found for {clean_id} (original: {record_id})")
        
        return label
        
    elif dataset_name == 'AFDB':
        # For AFDB, check the rhythm annotations.
        if 'annotations' in record_data:
            aux_notes = record_data['annotations']['aux_notes']
            # Convert to string if it's a list
            if isinstance(aux_notes, (list, np.ndarray)):
                aux_notes_str = ' '.join(str(note) for note in aux_notes if note)
            else:
                aux_notes_str = str(aux_notes) if aux_notes else ''
            
            if '(AFIB' in aux_notes_str:
                return 'A'  # AFib
            else:
                return 'N'  # Normal
        else:
            return 'N'  # Assume Normal if no annotations

In [31]:
# === DIAGNOSTIC: Test label lookup ===
print("\n" + "="*60)
print("TESTING LABEL LOOKUP")
print("="*60)

# Test with a few CINC records
test_records = cinc_records[:5]
for rec in test_records:
    clean_id = Path(rec).stem
    label = cinc_labels_dict.get(clean_id, 'UNKNOWN')
    print(f"Record: {rec}")
    print(f"  Clean ID: {clean_id}")
    print(f"  Label: {label}")
    print()

# Check overall distribution in dictionary
from collections import Counter
label_dist = Counter(cinc_labels_dict.values())
print(f"Label distribution in REFERENCE-v3.csv:")
for label, count in sorted(label_dist.items()):
    print(f"   {label}: {count}")
print("="*60 + "\n")


TESTING LABEL LOOKUP
Record: sample2017\validation\A00001.mat
  Clean ID: A00001
  Label: N

Record: sample2017\validation\A00002.mat
  Clean ID: A00002
  Label: N

Record: sample2017\validation\A00003.mat
  Clean ID: A00003
  Label: N

Record: sample2017\validation\A00004.mat
  Clean ID: A00004
  Label: A

Record: sample2017\validation\A00005.mat
  Clean ID: A00005
  Label: A

Label distribution in REFERENCE-v3.csv:
   A: 758
   N: 5076
   O: 2415
   ~: 279



Step 4: Put It All Together and Process All Data
Now you can loop through your master_list, apply your functions, and save all the clean windows and their corresponding labels.

In [32]:
all_windows = []
all_labels = []

# Loop through every record from both datasets
for record_id, loader, dataset_name in tqdm(master_list, desc="Processing all records"):
    try:
        # Load the raw data
        raw_data = loader.load_record(record_id)
        
        # Determine the label for the entire record
        label = get_label(record_id, raw_data, dataset_name)
        
        # We only care about AFib ('A') vs Normal ('N') for now
        if label not in ['A', 'N']:
            continue
            
        # Select a single lead (always use the first one for consistency)
        signal = raw_data['signal'][:, 0] if raw_data['signal'].ndim > 1 else raw_data['signal']
        
        # Process the signal into clean, uniform windows
        windows = preprocess_and_segment(signal, raw_data['fs'])
        
        if windows.shape[0] > 0:
            all_windows.append(windows)
            # Assign the same label to all windows from this record
            all_labels.extend([label] * len(windows))

    except Exception as e:
        print(f"⚠️  Skipping record {record_id} due to error: {e}")

# Concatenate all windows into a single large NumPy array
X = np.concatenate(all_windows, axis=0)
y = np.array(all_labels)

print(f"\n✅ Processing complete!")
print(f"   Total windows created: {len(X)}")
print(f"   Shape of X (data): {X.shape}")
print(f"   Shape of y (labels): {y.shape}")
print(f"   Normal windows: {np.sum(y == 'N')}")
print(f"   AFib windows: {np.sum(y == 'A')}")

# --- Save the final processed dataset ---
processed_dir = Path('../data/processed/')
processed_dir.mkdir(exist_ok=True)
np.save(processed_dir / 'X_data.npy', X)
np.save(processed_dir / 'y_labels.npy', y)

print(f"\n💾 Saved processed data to {processed_dir}")

Processing all records: 100%|██████████| 8851/8851 [00:36<00:00, 241.85it/s]



✅ Processing complete!
   Total windows created: 103193
   Shape of X (data): (103193, 2500)
   Shape of y (labels): (103193,)
   Normal windows: 16325
   AFib windows: 86868

💾 Saved processed data to ..\data\processed
